In [1]:
# imports and ensure versions

import numpy as np, pandas as pd, sklearn, xgboost, joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

# change seed value, different seed values may yield different results
SEED = 100

print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)

sklearn 1.9.1 | xgboost 3.4.1


In [2]:
# sampling config 
import os
from pathlib import Path


os.chdir(Path.home() / "Projects/ML-Based-NIDS")  # repo root; adjust if yours differs
print("cwd:", Path.cwd())

DATASETS = {
    "NF-UNSW-NB15-v2": 2_390_275,
    "NF-CSE-CIC-IDS2018-v2": 18_893_708,
}  # name -> total rows, from the UQ page
SAMPLE_SIZE = 200_000   # rows per dataset
SAMPLE_SEED = 42        # sampling seed (separate from the model SEED above)
CHUNK_SIZE = 1_000_000  # rows per chunk when reading the big CSVs
DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "processed"
REBUILD = False         # True = redo sampling even if the parquet already exists

cwd: /home/loqran/Projects/ML-Based-NIDS


In [3]:
# label, drop IPs, clean, and take a stratified random sample of each dataset
def sample_dataset(name, total_rows):
    out_path = OUT_DIR / f"{name}_sample.parquet"
    if out_path.exists() and not REBUILD:
        print(f"{name}: {out_path} exists, skipping (set REBUILD = True to redo)")
        return

    path = next(DATA_DIR.rglob(f"{name}.csv"), None)
    if path is None:
        raise FileNotFoundError(
            f"Could not find {name}.csv under {DATA_DIR}/ - extract the zip there first.")
    frac = SAMPLE_SIZE / total_rows
    parts, rows_read = [], 0

    # read in chunks: the CIC file is too big to load into memory at once
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        rows_read += len(chunk)
        chunk["Label"] = (chunk["Attack"] != "Benign").astype(int)  # benign = 0, attack = 1
        chunk = chunk.drop(columns=["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"])
        chunk = chunk.replace([np.inf, -np.inf], np.nan).dropna()
        # same fraction from each class in every chunk = stratified sample
        parts.append(chunk.groupby("Label").sample(frac=frac, random_state=SAMPLE_SEED))

    df = pd.concat(parts, ignore_index=True)
    duplicates = df.duplicated().sum()
    df = df.drop_duplicates()  # so no flow ends up in both the train and test split

    print(f"{name}: read {rows_read:,} rows (UQ page: {total_rows:,}), "
          f"removed {duplicates:,} duplicates, kept {len(df):,} rows, "
          f"{df['Label'].mean():.1%} attacks, {df.shape[1] - 1} features")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)

for name, total_rows in DATASETS.items():
    sample_dataset(name, total_rows)

NF-UNSW-NB15-v2: data/processed/NF-UNSW-NB15-v2_sample.parquet exists, skipping (set REBUILD = True to redo)
NF-CSE-CIC-IDS2018-v2: data/processed/NF-CSE-CIC-IDS2018-v2_sample.parquet exists, skipping (set REBUILD = True to redo)


In [4]:
# loading cell

unsw = pd.read_parquet("data/processed/NF-UNSW-NB15-v2_sample.parquet")
cic = pd.read_parquet("data/processed/NF-CSE-CIC-IDS2018-v2_sample.parquet")

X_unsw, y_unsw = unsw.drop(columns="Label"), unsw["Label"]
X_cic, y_cic = cic.drop(columns="Label"), cic["Label"]

features = sorted(set(X_unsw.columns) & set(X_cic.columns))
X_unsw, X_cic = X_unsw[features], X_cic[features]

limit = np.finfo(np.float32).max
X_unsw, X_cic = X_unsw.clip(-limit, limit), X_cic.clip(-limit, limit)

print(len(features), "features")
print("UNSW attack share:", y_unsw.mean().round(3), "| CIC:", y_cic.mean().round(3))


41 features
UNSW attack share: 0.037 | CIC: 0.12


In [5]:
#split cell
splits = {}
for name, X, y in [("UNSW", X_unsw, y_unsw), ("CIC", X_cic, y_cic)]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED)
    splits[name] = dict(X_tr=X_tr, X_te=X_te, y_tr=y_tr, y_te=y_te)

In [6]:
# new model each time this is called
# currently 3 models, Random Fores, XGB Classifier, and Logistic Regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression

def make_models():
    return {
        "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
        "XGB": XGBClassifier(n_estimators=300, tree_method="hist",
                             n_jobs=-1, random_state=SEED, eval_metric="logloss"),
        "LR":  make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                             LogisticRegression(max_iter=1000, random_state=SEED)),
    }

In [7]:
# recall, fpr, f1
def evaluate(model, X, y):
    pred = model.predict(X)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {"recall": recall_score(y, pred),
    "FPR": fp / (fp + tn),
    "F1": f1_score(y, pred)}

In [8]:
# train the models twice, once on UNSW and once on CIC
rows, trained = [], {}
for train_on in ["UNSW", "CIC"]:
    for model_name, model in make_models().items():
        model.fit(splits[train_on]["X_tr"], splits[train_on]["y_tr"])
        trained[(model_name, train_on)] = model
        for test_on in ["UNSW", "CIC"]:
            m = evaluate(model, splits[test_on]["X_te"], splits[test_on]["y_te"])
            rows.append({"model": model_name, "train": train_on, "test": test_on, **m})
        print("done:", model_name, "trained on", train_on)

results = pd.DataFrame(rows)

done: RF trained on UNSW
done: XGB trained on UNSW
done: LR trained on UNSW
done: RF trained on CIC
done: XGB trained on CIC
done: LR trained on CIC


In [9]:
# print the results
Path("results").mkdir(exist_ok=True)

for model_name in ["RF", "XGB", "LR"]:
    print(f"\n{model_name}: F1 (rows = trained on, cols = tested on)")
    sub = results[results.model == model_name]
    print(sub.pivot(index="train", columns="test", values="F1").round(3))

results.to_csv("results/step2_results.csv", index=False)
for (m, d), model in trained.items():
    joblib.dump(model, f"results/{m}_{d}.joblib")


RF: F1 (rows = trained on, cols = tested on)
test     CIC   UNSW
train              
CIC    0.981  0.000
UNSW   0.058  0.944

XGB: F1 (rows = trained on, cols = tested on)
test     CIC   UNSW
train              
CIC    0.980  0.044
UNSW   0.096  0.937

LR: F1 (rows = trained on, cols = tested on)
test     CIC   UNSW
train              
CIC    0.928  0.049
UNSW   0.288  0.915


In [10]:
# prettyfy the output from results

results.pivot_table(index=["model", "train"], columns="test",
                    values=["recall", "FPR", "F1"]).round(3)

F1           FPR        recall       
test           CIC   UNSW    CIC   UNSW    CIC   UNSW
model train                                          
LR    CIC    0.928  0.049  0.006  0.037  0.902  0.050
      UNSW   0.288  0.915  0.413  0.005  0.678  0.960
RF    CIC    0.981  0.000  0.000  0.001  0.963  0.000
      UNSW   0.058  0.944  0.530  0.003  0.147  0.968
XGB   CIC    0.980  0.044  0.000  0.004  0.963  0.025
      UNSW   0.096  0.937  0.467  0.003  0.224  0.956